In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ROOT
from math import lgamma

# --------------------- Load CSV ---------------------
def load_csv(csv_path):
    df = pd.read_csv(csv_path)
    t, L, Eavg, alpha = df.iloc[:,0].to_numpy(), df.iloc[:,1].to_numpy(), df.iloc[:,2].to_numpy(), df.iloc[:,3].to_numpy()
    mask = np.isfinite(t) & np.isfinite(L) & np.isfinite(Eavg) & np.isfinite(alpha)
    t, L, Eavg, alpha = t[mask], L[mask], Eavg[mask], alpha[mask]
    idx = np.argsort(t)
    return t[idx], L[idx], Eavg[idx], alpha[idx]

# --------------------- Gamma PDF ---------------------
def gamma_pdf(E, mu, alpha):
    E = np.asarray(E)[:, None]
    k = 1 + alpha[None, :]
    theta = mu[None, :] / k
    lgk = np.vectorize(lgamma)(k)
    pdf = np.exp((k-1)*np.log(E) - E/theta - lgk - k*np.log(theta))
    return np.where(E>=0, pdf, 0)

# --------------------- Model PDF ---------------------
def model_pdf(t, L, Eavg, alpha, t_edges=1201, E_edges=1601, t_min=-0.05, t_max=10, E_min=0.0, E_max=120, nsig=8):
    # Time grid
    t0, t1 = t[0], t[-1]
    t_min = t_min if t_min is not None else t0
    t_max = t_max if t_max is not None else t1
    t_edges_arr = np.linspace(t_min, t_max, t_edges)
    t_cent = 0.5*(t_edges_arr[:-1]+t_edges_arr[1:])

    # Energy grid
    mu_c = np.interp(t_cent, t, Eavg)
    alpha_c = np.interp(t_cent, t, alpha)
    if E_max is None:
        E_max = max(np.max(mu_c + nsig*mu_c/np.sqrt(1+alpha_c)), 10*np.max(mu_c))
    E_edges_arr = np.linspace(E_min, E_max, E_edges)
    E_cent = 0.5*(E_edges_arr[:-1]+E_edges_arr[1:])

    # Compute model PDF
    L_c = np.interp(t_cent, t, L)
    R_c = L_c / mu_c
    p_t = R_c / np.trapz(R_c, t_cent)
    fEt = gamma_pdf(E_cent, mu_c, alpha_c)
    Z_model = fEt * p_t[None, :]

    # Normalize to window (so integral over dE dt = 1)
    dE, dt = np.diff(E_edges_arr), np.diff(t_edges_arr)
    area = dE[:,None] * dt[None,:]
    Z_model /= np.sum(Z_model * area)

    return t_edges_arr, E_edges_arr, t_cent, E_cent, Z_model

# --------------------- Plot heatmap and integrals ---------------------
def plot_model_and_integrals(csv_path):
    t, L, Eavg, alpha = load_csv(csv_path)
    max_flux = np.max(L)

    t_edges, E_edges, t_cent, E_cent, Z_model = model_pdf(t, L, Eavg, alpha)

    # Integrate over energy -> flux vs time
    dE = np.diff(E_edges)
    flux_vs_time = (Z_model.T * dE).T.sum(axis=0)
    scale = max_flux / np.max(flux_vs_time)
    Z_model_scaled = Z_model * scale
    flux_vs_time_scaled = flux_vs_time * scale

    # Integrate over time -> counts vs energy
    dt = np.diff(t_edges)
    counts_vs_energy = (Z_model_scaled * dt).sum(axis=1)

    # --- Plot 2D heatmap ---
    plt.figure(figsize=(8,5))
    plt.pcolormesh(t_edges, E_edges, Z_model_scaled, shading="auto", cmap="plasma")
    plt.colorbar(label="Flux (scaled to CSV peak)")
    plt.xlabel("Time")
    plt.ylabel("Energy")
    plt.title("Model PDF (scaled)")
    plt.show()

    # --- Plot integrated over energy -> flux vs time ---
    plt.figure(figsize=(7,4))
    plt.plot(t_cent, flux_vs_time_scaled, marker='.', linestyle='-')
    plt.xlabel("Time")
    plt.ylabel("Flux")
    plt.title("Flux vs Time (integrated over energy)")
    plt.grid(True)
    plt.show()

    # --- Plot integrated over time -> counts vs energy ---
    plt.figure(figsize=(7,4))
    plt.plot(E_cent, counts_vs_energy, marker='.', linestyle='-')
    plt.xlabel("Energy")
    plt.ylabel("Counts")
    plt.title("Counts vs Energy (integrated over time)")
    plt.grid(True)
    plt.show()

    # --- Convert counts vs energy to TH1D (PyROOT) ---
    hist_name = "counts_vs_energy"
    nbins = len(counts_vs_energy)

    # Ensure edges are double precision for ROOT
    edges_array = np.array(E_edges, dtype=np.float64)

    # Create histogram with variable bin widths
    th1 = ROOT.TH1D(hist_name, "Counts vs Energy (Integrated Model Flux)", nbins, edges_array)

    # Fill histogram bin contents
    for i in range(nbins):
        th1.SetBinContent(i + 1, counts_vs_energy[i])

    # --- Save ROOT file ---
    output = ROOT.TFile("counts_vs_energy_Nakazato.root", "RECREATE")
    th1.Write()
    output.Close()

    print("✅ Saved counts_vs_energy_Nakazato.root (PyROOT TH1D)")

# --------------------- Run ---------------------
csv_path = "Nakazato_nu_e_timeseries.csv"
t_edges, E_edges, Z_model_scaled, flux_vs_time_scaled, counts_vs_energy, th1 = plot_model_and_integrals(csv_path)


Welcome to JupyROOT 6.28/04


FileNotFoundError: [Errno 2] No such file or directory: 'Nakazato_nu_e_timeseries.csv'